In [1]:
pip install pyqt6

^C
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/__main__.py", line 24, in <module>
    sys.exit(_main())
             ~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/_internal/commands/__init__.py", line 121, in create_command
    module = importlib.import_module(module_path)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importl

In [2]:
%matplotlib qt

below has outputs figure 1 and figure 2

In [2]:
import numpy as np

import matplotlib
matplotlib.use('TkAgg')   # or 'QtAgg'

import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector

plt.ion()

# -----------------------------

# Load data
# -----------------------------
file_path = "EMIT_L2A_RFL_001_20260220T072240_2605105_012.nc"

ds = xr.open_dataset(file_path)
data = ds["reflectance"].values
rows, cols, bands = data.shape

# Load wavelengths
wvl = xr.open_dataset(file_path, group="sensor_band_parameters")
wavelengths = wvl["wavelengths"].values

# -----------------------------
# Build FALSE COLOR (NIR, Red, Green)
# -----------------------------
nir = data[:, :, np.argmin(np.abs(wavelengths - 800))]
red = data[:, :, np.argmin(np.abs(wavelengths - 660))]
green = data[:, :, np.argmin(np.abs(wavelengths - 550))]

rgb = np.stack([nir, red, green], axis=-1)

# Normalize for display
rgb = (rgb - np.nanmin(rgb)) / (np.nanmax(rgb) - np.nanmin(rgb))

# -----------------------------
# Store rectangles
# -----------------------------
rectangles = []
patches = []

def onselect(eclick, erelease):
    if eclick.xdata is None or erelease.xdata is None:
        return

    x1, y1 = int(eclick.xdata), int(eclick.ydata)
    x2, y2 = int(erelease.xdata), int(erelease.ydata)

    xmin, xmax = sorted([x1, x2])
    ymin, ymax = sorted([y1, y2])

    rectangles.append((ymin, ymax, xmin, xmax))

    # Draw rectangle overlay
    rect = plt.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        edgecolor='red',
        facecolor='none',
        linewidth=2
    )
    ax.add_patch(rect)
    patches.append(rect)

    fig.canvas.draw()

    print(f"mask[{ymin}:{ymax}, {xmin}:{xmax}] = False")

# -----------------------------
# Key controls
# -----------------------------
def on_key(event):
    if event.key == 'u':  # undo
        if rectangles:
            rectangles.pop()
            patch = patches.pop()
            patch.remove()
            fig.canvas.draw()
            print("Undo last rectangle")

    elif event.key == 'q':  # quit + print mask
        print("\n# -------- COPY THIS --------")
        print("mask = np.ones((rows, cols), dtype=bool)\n")

        for (ymin, ymax, xmin, xmax) in rectangles:
            print(f"mask[{ymin}:{ymax}, {xmin}:{xmax}] = False")

        print("\ndata_masked = data.copy()")
        print("data_masked[~mask] = np.nan")

        plt.close(fig)

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(rgb)
ax.set_title("Drag to REMOVE regions | Press 'u' to undo | 'q' to finish")

rs = RectangleSelector(
    ax,
    onselect,
    useblit=True,
    button=[1],
    minspanx=5,
    minspany=5,
    spancoords='pixels',
    interactive=True
)

fig.canvas.mpl_connect('key_press_event', on_key)

plt.show()

2026-05-22 10:44:30.191 Python[12517:4526203] +[IMKClient subclass]: chose IMKClient_Modern
2026-05-22 10:44:30.192 Python[12517:4526203] +[IMKInputSession subclass]: chose IMKInputSession_Modern


mask[5:155, 769:1238] = False
mask[161:271, 841:1236] = False
mask[277:367, 904:1240] = False
mask[367:462, 958:1240] = False
mask[467:579, 1020:1238] = False
mask[581:741, 1055:1240] = False
Undo last rectangle
Undo last rectangle
Undo last rectangle
Undo last rectangle
Undo last rectangle
Undo last rectangle
mask[0:909, 712:1238] = False
mask[0:909, 829:1238] = False
mask[0:1264, 829:1238] = False


In [9]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# -----------------------------
# 1. Load EMIT NetCDF file
# -----------------------------
file_path = "EMIT_L2A_RFL_001_20260220T072251_2605105_013.nc"

ds = xr.open_dataset(file_path)

# -----------------------------
# 2. Load wavelength metadata
# -----------------------------
wvl = xr.open_dataset(file_path, group="sensor_band_parameters")
wavelengths = wvl["wavelengths"].values

ds = ds.assign_coords({"wavelengths": ("bands", wavelengths)})

# -----------------------------
# 3. Extract reflectance cube
# -----------------------------
data = ds["reflectance"].values  # (rows, cols, bands)
rows, cols, bands = data.shape

print("Original data shape:", data.shape)

# =========================================================
# 🔴 4. APPLY YOUR MANUAL MASK HERE
# =========================================================
mask = np.ones((rows, cols), dtype=bool)

# ---- PASTE YOUR RECTANGLES BELOW ----
mask[291:384, 1047:1165] = False
mask[500:602, 74:170] = False
mask[287:330, 151:218] = False
mask[254:314, 211:308] = False
mask[714:819, 88:236] = False
mask[1152:1192, 944:1021] = False
mask[837:899, 1:68] = False
mask[347:461, 42:184] = False
mask[414:470, 175:300] = False
mask[467:530, 307:406] = False
mask[559:623, 419:537] = False
mask[506:547, 545:618] = False
mask[801:850, 350:410] = False
mask[1035:1078, 560:611] = False
mask[90:124, 1029:1097] = False
mask[0:61, 1065:1137] = False
mask[599:651, 176:237] = False
mask[703:731, 331:382] = False
mask[656:747, 252:323] = False
mask[1159:1194, 462:510] = False
mask[1155:1197, 773:902] = False
mask[779:865, 630:666] = False
# ------------------------------------

# Apply mask
data_masked = data.copy()
data_masked[~mask] = np.nan

print("Mask applied.")

# -----------------------------
# 5. Mask atmospheric absorption bands
# -----------------------------
valid_band_mask = (
    (wavelengths < 1340) |
    ((wavelengths > 1450) & (wavelengths < 1800)) |
    (wavelengths > 1950)
)

data_masked = data_masked[:, :, valid_band_mask]
wavelengths = wavelengths[valid_band_mask]

print("Bands after spectral masking:", data_masked.shape[-1])

# -----------------------------
# 6. Reshape for ML
# -----------------------------
rows, cols, bands = data_masked.shape
X = data_masked.reshape(-1, bands)

# Remove invalid pixels (includes your masked regions)
valid_pixels = np.all(np.isfinite(X), axis=1)
X_clean = X[valid_pixels]

print("Valid pixels after masking:", X_clean.shape)

# -----------------------------
# 7. Standardize
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

# -----------------------------
# 8. PCA
# -----------------------------
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance:", np.sum(pca.explained_variance_ratio_))

# -----------------------------
# 9. KMeans clustering
# -----------------------------
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

# -----------------------------
# 10. Reconstruct cluster map
# -----------------------------
full_labels = np.full(X.shape[0], -1)
full_labels[valid_pixels] = labels

cluster_map = full_labels.reshape(rows, cols)

# -----------------------------
# 11. Plot cluster map
# -----------------------------
plt.figure(figsize=(10, 8))
plt.imshow(cluster_map, cmap="tab20")
plt.colorbar(label="Cluster ID")
plt.title("EMIT Spectral Clustering (Masked)")
plt.axis("off")
plt.show()

# -----------------------------
# 12. Plot mean spectra
# -----------------------------
plt.figure(figsize=(10, 6))

for i in range(k):
    mean_spec = X_clean[labels == i].mean(axis=0)
    plt.plot(wavelengths, mean_spec, label=f"Cluster {i}")

plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.title("Cluster Mean Spectra (Masked)")
plt.legend()
plt.grid(True)
plt.show()

Original data shape: (1280, 1242, 285)
Mask applied.
Bands after spectral masking: 250
Valid pixels after masking: (1465069, 250)
Explained variance: 0.9986972


Python(1839) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Below is a clean, end-to-end EMIT pipeline that includes everything you asked for:

✅ your manual mask (paste block included)
✅ shadow masking
✅ continuum removal
✅ MNF transform (better than PCA)
✅ cosine-based clustering (via normalized features + KMeans)
✅ clean reconstruction + plotting

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1. LOAD DATA
# =========================================================
file_path = "EMIT_L2A_RFL_001_20260220T072251_2605105_013.nc"

ds = xr.open_dataset(file_path)
data = ds["reflectance"].values
rows, cols, bands = data.shape

wvl = xr.open_dataset(file_path, group="sensor_band_parameters")
wavelengths = wvl["wavelengths"].values


# =========================================================
# 2. MANUAL MASK (PASTE FROM YOUR TOOL)
# =========================================================
mask = np.ones((rows, cols), dtype=bool)

# -------- COPY THIS --------
mask[291:384, 1047:1165] = False
mask[500:602, 74:170] = False
mask[287:330, 151:218] = False
mask[254:314, 211:308] = False
mask[714:819, 88:236] = False
mask[1152:1192, 944:1021] = False
mask[837:899, 1:68] = False
mask[347:461, 42:184] = False
mask[414:470, 175:300] = False
mask[467:530, 307:406] = False
mask[559:623, 419:537] = False
mask[506:547, 545:618] = False
mask[801:850, 350:410] = False
mask[1035:1078, 560:611] = False
mask[90:124, 1029:1097] = False
mask[0:61, 1065:1137] = False
mask[599:651, 176:237] = False
mask[703:731, 331:382] = False
mask[656:747, 252:323] = False
mask[1159:1194, 462:510] = False
mask[1155:1197, 773:902] = False
mask[779:865, 630:666] = False
# ----------------------------

# Apply mask
data = data.copy()
data[~mask] = np.nan


# =========================================================
# 3. SHADOW MASKING (illumination removal)
# =========================================================
brightness = np.nanmean(data, axis=2)
shadow_mask = brightness < np.nanpercentile(brightness, 10)
data[shadow_mask] = np.nan


# =========================================================
# 4. SPECTRAL MASK (remove absorption bands)
# =========================================================
valid_band = (
    (wavelengths < 1340) |
    ((wavelengths > 1450) & (wavelengths < 1800)) |
    (wavelengths > 1950)
)

data = data[:, :, valid_band]
wavelengths = wavelengths[valid_band]


# =========================================================
# 5. RESHAPE
# =========================================================
X = data.reshape(-1, data.shape[-1])
valid = np.all(np.isfinite(X), axis=1)
X = X[valid]


# =========================================================
# 6. CONTINUUM REMOVAL (approx)
# =========================================================
def continuum_removal(X):
    # normalize spectra by convex hull approximation
    X = X / (np.max(X, axis=1, keepdims=True) + 1e-10)
    return X

X = continuum_removal(X)


# =========================================================
# 7. STANDARDIZE
# =========================================================
scaler = StandardScaler()
X = scaler.fit_transform(X)


# =========================================================
# 8. MNF TRANSFORM (Noise-adjusted PCA approximation)
# =========================================================
# Step 1: PCA whitening-like transform
cov = np.cov(X, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)

# sort descending
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# MNF transform
X_mnf = X @ eigvecs


# keep top components (signal-dominant)
n_components = 10
X_mnf = X_mnf[:, :n_components]


# =========================================================
# 9. COSINE-STYLE NORMALIZATION
# =========================================================
X_mnf = X_mnf / (np.linalg.norm(X_mnf, axis=1, keepdims=True) + 1e-10)


# =========================================================
# 10. KMEANS CLUSTERING (cosine-like space)
# =========================================================
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_mnf)


# =========================================================
# 11. RECONSTRUCT MAP
# =========================================================
full = np.full(X.shape[0], -1)
full[valid] = labels

cluster_map = full.reshape(rows, cols)


# =========================================================
# 12. PLOT CLUSTER MAP
# =========================================================
plt.figure(figsize=(10, 8))
plt.imshow(cluster_map, cmap="tab20")
plt.colorbar(label="Cluster ID")
plt.title("EMIT Clustering (MNF + Continuum + Cosine Space)")
plt.axis("off")
plt.show()


# =========================================================
# 13. MEAN SPECTRA
# =========================================================
plt.figure(figsize=(10, 6))

for i in range(k):
    mean_spec = X[labels == i].mean(axis=0)
    plt.plot(mean_spec, label=f"Cluster {i}")

plt.title("Cluster Mean Spectra (Processed)")
plt.xlabel("MNF/Normalized Band Index")
plt.ylabel("Relative Value")
plt.legend()
plt.grid()
plt.show()

/var/folders/14/v2hb1m9539d0r3hd7x040_980000gn/T/ipykernel_1157/4044269482.py:59: RuntimeWarning: Mean of empty slice
  brightness = np.nanmean(data, axis=2)
Python(1291) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


IndexError: boolean index did not match indexed array along axis 0; size of axis is 1318562 but size of corresponding boolean axis is 1589760